# Task 3 — LightGBM point forecast

Two versions of the same model, with the same calendar and lag features and the same hyperparameters.

1. `lgbm_martina_noweather` — calendar and lags, with no weather and without the Terna column.
2. `lgbm_martina_obs_upper_bound` — the same features, plus the six ERA5 temperatures and degree-hours derived from those temperatures at the target hour. This is an upper bound: ERA5 at the target hour was not known at 10:00 on the previous day.

The forecast file is not available. This run does not download it, does not replace it with ERA5, and does not write `lgbm_fc_gfs` predictions.

2025 is not used for training, for choosing hyperparameters, or for the score. Hyperparameters were chosen only on local-day folds inside the training set, and that search is not repeated here.

The official train and validation splits stay as they are. Lags are built on the full hourly series. The final fit of both models keeps only training targets already published at the first 2024 forecast cutoff, 2023-12-31 09:00 UTC. An hour starting at `t` is treated as published at `t + 2 hours`, and it is kept when that time is at or before the cutoff.


In [1]:
from pathlib import Path
import sys

import lightgbm as lgb
import numpy as np
import pandas as pd

for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "split_config.py").exists():
        sys.path.insert(0, str(candidate))
        REPO = candidate
        break
else:
    raise FileNotFoundError("split_config.py not found from the current directory.")

from split_config import (
    FORECAST_COVERAGE_VERIFIED,
    LOCAL_TZ,
    OBSERVED_TEMP_COLUMNS,
    PUBLICATION_LAG_HOURS,
    TARGET_COL,
    TERNA_COL,
    add_lags,
    check_predictions,
    cutoff_timestamps,
    load_data,
    load_observed_weather,
    metrics,
    seasonal_average_prediction,
    seasonal_naive,
    split,
    validation_index,
)

if FORECAST_COVERAGE_VERIFIED:
    raise RuntimeError("Forecast coverage is still closed in this notebook.")

df = load_data()
lagged = add_lags(df)
train, val = split(lagged)
print("train", len(train), "val", len(val))
print("forecast coverage verified:", FORECAST_COVERAGE_VERIFIED)
print("val years", sorted(set(val.index.tz_convert(LOCAL_TZ).year)))


train 17160 val 8784
forecast coverage verified: False
val years [2024]


## Features

Local hour and day of week enter as sine and cosine. `is_holiday` stays the CSV flag, including the November Sundays already in the file.

A holiday eve is `is_holiday` on the next local day. A bridge day is a weekday that is not itself a holiday and that sits between two holidays, or between a holiday and the weekend. These are calendar labels, not a second holiday list. The eve of 31 December 2024 uses only the `is_holiday` flag of 1 January, not the 2025 load.

The lags are the UTC shifts of 48, 168 and 336 hours. Degree-hours in the upper-bound version use the mean of the six temperatures at the target hour, with a base of 18 °C. There is no rolling mean.


In [2]:
DEGREE_BASE_C = 18.0
COMMON_FEATURES = [
    "hour_sin",
    "hour_cos",
    "dow_sin",
    "dow_cos",
    "month",
    "is_weekend",
    "is_holiday",
    "is_summer",
    "is_christmas_period",
    "is_holiday_eve",
    "is_bridge",
    "actual_load_mw_lag48h",
    "actual_load_mw_lag168h",
    "actual_load_mw_lag336h",
]
WEATHER_FEATURES = [
    *OBSERVED_TEMP_COLUMNS,
    "heating_degree_c",
    "cooling_degree_c",
]


def add_calendar_indicators(frame: pd.DataFrame) -> pd.DataFrame:
    """Holiday eve and bridge days from the CSV flags. No second holiday list."""
    out = frame.copy()
    local_day = out.index.tz_convert(LOCAL_TZ).floor("D")
    daily = (
        pd.DataFrame(
            {
                "is_holiday": out["is_holiday"].to_numpy(),
                "is_weekend": out["is_weekend"].to_numpy(),
                "dow": out["day_of_week"].to_numpy(),
            },
            index=local_day,
        )
        .groupby(level=0)
        .first()
        .sort_index()
    )
    next_holiday = daily["is_holiday"].shift(-1)
    prev_holiday = daily["is_holiday"].shift(1)
    next_weekend = daily["is_weekend"].shift(-1)
    prev_weekend = daily["is_weekend"].shift(1)
    bridge = (
        daily["dow"].lt(5)
        & daily["is_holiday"].eq(0)
        & (
            (prev_holiday.eq(1) & next_holiday.eq(1))
            | (prev_holiday.eq(1) & next_weekend.eq(1))
            | (prev_weekend.eq(1) & next_holiday.eq(1))
        )
    )
    out["is_holiday_eve"] = next_holiday.fillna(0).astype(int).loc[local_day].to_numpy()
    out["is_bridge"] = bridge.astype(int).loc[local_day].to_numpy()
    out["hour_sin"] = np.sin(2 * np.pi * out["hour_local"] / 24)
    out["hour_cos"] = np.cos(2 * np.pi * out["hour_local"] / 24)
    out["dow_sin"] = np.sin(2 * np.pi * out["day_of_week"] / 7)
    out["dow_cos"] = np.cos(2 * np.pi * out["day_of_week"] / 7)
    return out


def observed_temperature_features(index, weather: pd.DataFrame) -> pd.DataFrame:
    """Same-hour ERA5 temperatures and degree-hours. Upper bound only."""
    temps = weather.reindex(index)[list(OBSERVED_TEMP_COLUMNS)]
    if temps.isna().any().any():
        raise ValueError("ERA5 temperatures do not cover every target hour.")
    mean_temp = temps.mean(axis=1)
    temps = temps.copy()
    temps["heating_degree_c"] = (DEGREE_BASE_C - mean_temp).clip(lower=0)
    temps["cooling_degree_c"] = (mean_temp - DEGREE_BASE_C).clip(lower=0)
    return temps


lagged = add_calendar_indicators(lagged)
train, val = split(lagged)
weather = load_observed_weather()
train_weather = observed_temperature_features(train.index, weather)
val_weather = observed_temperature_features(val.index, weather)

used = pd.concat([train, val])
if not np.isfinite(used[COMMON_FEATURES].to_numpy(dtype=float)).all():
    raise ValueError("A calendar or lag feature is missing on train or validation.")
if train.index.tz_convert(LOCAL_TZ).year.max() > 2023:
    raise ValueError("Training rows go past local year 2023.")
if set(val.index.tz_convert(LOCAL_TZ).year) != {2024}:
    raise ValueError("Validation is not local year 2024.")

print("common features", len(COMMON_FEATURES))
print("upper-bound features", len(COMMON_FEATURES) + len(WEATHER_FEATURES))
print("holiday eves in validation", int(val["is_holiday_eve"].sum()))
print("bridge hours in validation", int(val["is_bridge"].sum()))


common features 14
upper-bound features 22
holiday eves in validation 287
bridge hours in validation 72


## Hyperparameters

One grid was scored on the no-weather features. Each candidate was evaluated on the four expanding local-day folds of the training set. The criterion was the MAE pooled over all the hours in those folds (not the mean of per-fold MAEs). 2024 was not used for the choice. The chosen hyperparameters are used for the ERA5 version as well. The search is not run again.

The grid is wider than a first pass would suggest: the best candidate of a narrower grid sat on its edge for `n_estimators` and `learning_rate`, so the range extends beyond those values, and `num_leaves` goes below 31. The selected values are 800 trees, learning rate 0.2, and 15 leaves. Each one sits on the edge of this grid, and the cell below prints that warning.

In [3]:
GRID_VALUES = {
    "n_estimators": (200, 400, 800),
    "learning_rate": (0.05, 0.1, 0.2),
    "num_leaves": (15, 31, 63),
}

# Selected by pooled MAE on the four expanding local-day folds of the
# official training split. Reused as fixed settings; the search is not repeated.
best = {
    "n_estimators": 800,
    "learning_rate": 0.2,
    "num_leaves": 15,
}
print("chosen", best)

for name, values in GRID_VALUES.items():
    if best[name] in (min(values), max(values)):
        print(f"warning: chosen {name}={best[name]} is on the edge of the grid {values}")

chosen {'n_estimators': 800, 'learning_rate': 0.2, 'num_leaves': 15}


## Fit and 2024 validation forecast

Each version is refit with the hyperparameters already fixed, and predicts every hour of 2024.

The official training split still contains every local hour from 16 January 2022 through 31 December 2023. Those rows stay in the series, so lag features are unchanged. The fitting rows are the subset whose target was already published at the first 2024 cutoff. Publication time is the timestamp plus 2 hours. A row is kept when that time is at or before 2023-12-31 09:00 UTC. The check below requires exactly 17,145 fitting rows.


In [4]:
def fit_model(x_train, y_train):
    model = lgb.LGBMRegressor(
        **best,
        objective="regression",
        random_state=0,
        verbosity=-1,
        n_jobs=-1,
    )
    model.fit(x_train, y_train)
    return model


# 10:00 Europe/Rome on the day before the first validation hour.
first_validation_cutoff = cutoff_timestamps(val.index[:1])[0].tz_convert("UTC")
expected_cutoff = pd.Timestamp("2023-12-31 09:00:00", tz="UTC")
if first_validation_cutoff != expected_cutoff:
    raise ValueError(f"Unexpected first 2024 cutoff: {first_validation_cutoff}")

published_at = train.index + pd.Timedelta(hours=PUBLICATION_LAG_HOURS)
final_fit_mask = published_at <= first_validation_cutoff
fit_train = train.loc[final_fit_mask]
fit_weather = train_weather.loc[final_fit_mask]

if len(train) != 17_160 or len(val) != 8_784:
    raise ValueError("The official train or validation split changed.")
if len(fit_train) != 17_145:
    raise ValueError(f"Expected 17145 published training targets, found {len(fit_train)}.")
if fit_train.index.tz_convert(LOCAL_TZ).year.max() > 2023:
    raise ValueError("The final fit includes a local year after 2023.")

# Hours that fail the rule stay in the original series and can still be lags.
lag_source = fit_train.index.max() + pd.Timedelta(hours=1)
lag_user = lag_source + pd.Timedelta(hours=48)
if lag_source in fit_train.index:
    raise ValueError("A target unpublished at the first cutoff was used for fitting.")
if not np.isclose(val.loc[lag_user, "actual_load_mw_lag48h"], df.loc[lag_source, TARGET_COL]):
    raise ValueError("Lag features were not taken from the original series.")

print("first cutoff", first_validation_cutoff)
print("official training rows", len(train))
print("rows used for fitting", len(fit_train))
print("rows excluded by the publication rule", int((~final_fit_mask).sum()))
print("last training target", fit_train.index.max())

model_noweather = fit_model(fit_train[COMMON_FEATURES], fit_train[TARGET_COL])
pred_noweather = model_noweather.predict(val[COMMON_FEATURES])

x_train_obs = pd.concat([fit_train[COMMON_FEATURES], fit_weather[WEATHER_FEATURES]], axis=1)
x_val_obs = pd.concat([val[COMMON_FEATURES], val_weather[WEATHER_FEATURES]], axis=1)
model_obs = fit_model(x_train_obs, fit_train[TARGET_COL])
pred_obs = model_obs.predict(x_val_obs)

if len(pred_noweather) != len(val) or len(pred_obs) != len(val):
    raise ValueError("A model did not score every validation hour.")
print("validation hours", len(val))


first cutoff 2023-12-31 09:00:00+00:00
official training rows 17160
rows used for fitting 17145
rows excluded by the publication rule 15
last training target 2023-12-31 07:00:00+00:00
validation hours 8784


## CSV files and comparison

The three references are the same-local-hour naive forecast, the hour-of-week mean estimated on the official training split, and Terna's forecast. Terna is not a feature. The reference calculations are unchanged: only the two LightGBM fits drop targets that were still unpublished at the first 2024 cutoff.

The CSV files hold every hour of 2024. `check_predictions` accepted both files, 8,784 rows each, with `y_true` equal to `actual_load_mw`. Metrics are reported on all 8,784 validation hours, and again on the 8,783 hours where the seasonal naive exists. The omitted hour is `2024-04-07 00:00 UTC`: the same Europe/Rome clock time seven days earlier did not exist, because 31 March springs forward at 02:00.

The final fit uses 17,145 training targets. The last included hour is `2023-12-31 07:00 UTC`, published at the cutoff. The next training hour, `2023-12-31 08:00 UTC`, is published at 10:00 UTC and is excluded. The official split is still 17,160 training hours and 8,784 validation hours.

MAE and RMSE are in MW.

All 8,784 validation hours:

| Model | n | MAE | RMSE |
|---|---:|---:|---:|
| `lgbm_martina_noweather` | 8,784 | 1,102.40 | 1,547.59 |
| `lgbm_martina_obs_upper_bound` | 8,784 | 998.45 | 1,409.79 |
| `seasonal_avg` | 8,784 | 2,718.46 | 3,653.91 |
| `terna` | 8,784 | 603.93 | 816.30 |

Common 8,783 hours:

| Model | n | MAE | RMSE |
|---|---:|---:|---:|
| `lgbm_martina_noweather` | 8,783 | 1,102.52 | 1,547.68 |
| `lgbm_martina_obs_upper_bound` | 8,783 | 998.56 | 1,409.87 |
| `seasonal_naive` | 8,783 | 1,975.14 | 3,177.19 |
| `seasonal_avg` | 8,783 | 2,718.58 | 3,654.08 |
| `terna` | 8,783 | 603.98 | 816.34 |

Both LightGBM models remain ahead of the seasonal references. Terna's forecast is still the most accurate of the five. The ERA5 version is lower error than the no-weather version and is still an upper bound: those temperatures were not known at 10:00 on the previous day.


In [5]:
OUT_DIR = REPO / "notebooks" / "martina" / "predictions"
OUT_DIR.mkdir(parents=True, exist_ok=True)


def prediction_frame(y_pred, model_name) -> pd.DataFrame:
    return pd.DataFrame(
        {
            "timestamp": val.index.tz_convert("UTC").strftime("%Y-%m-%dT%H:%M:%SZ"),
            "y_true": val[TARGET_COL].to_numpy(),
            "y_pred": y_pred,
            "model": model_name,
        }
    )


paths = {
    "lgbm_martina_noweather": OUT_DIR / "lgbm_martina_noweather.csv",
    "lgbm_martina_obs_upper_bound": OUT_DIR / "lgbm_martina_obs_upper_bound.csv",
}
prediction_frame(pred_noweather, "lgbm_martina_noweather").to_csv(
    paths["lgbm_martina_noweather"], index=False
)
prediction_frame(pred_obs, "lgbm_martina_obs_upper_bound").to_csv(
    paths["lgbm_martina_obs_upper_bound"], index=False
)

expected = validation_index(df)
for name, path in paths.items():
    checked = check_predictions(path, expected, reference_y=df[TARGET_COL])
    print(name, "checked rows", len(checked))

history = df.loc[: val.index.max()]
naive = seasonal_naive(history).reindex(val.index)
seasonal_avg = pd.Series(np.asarray(seasonal_average_prediction(train, val), dtype=float), index=val.index)
terna = val[TERNA_COL]

candidates = {
    "lgbm_martina_noweather": pd.Series(pred_noweather, index=val.index),
    "lgbm_martina_obs_upper_bound": pd.Series(pred_obs, index=val.index),
    "seasonal_naive": naive,
    "seasonal_avg": seasonal_avg,
    "terna": terna,
}

full_rows = [
    {"scope": "all_8784", "model": name, **metrics(val[TARGET_COL], prediction)}
    for name, prediction in candidates.items()
    if prediction.notna().all()
]
common = val.index[naive.notna()]
if len(common) != 8_783:
    raise ValueError(f"Expected 8783 common comparison hours, found {len(common)}.")
y_common = val.loc[common, TARGET_COL]
common_rows = [
    {"scope": "common_8783", "model": name, **metrics(y_common, prediction.loc[common])}
    for name, prediction in candidates.items()
]
comparison = pd.DataFrame(full_rows + common_rows)[["scope", "model", "n", "MAE", "RMSE"]]
print("hours in the common comparison", len(common), "of", len(val))
print(comparison.to_string(index=False, float_format=lambda value: f"{value:,.2f}"))

lgbm_martina_noweather checked rows 8784
lgbm_martina_obs_upper_bound checked rows 8784
hours in the common comparison 8783 of 8784
      scope                        model    n      MAE     RMSE
   all_8784       lgbm_martina_noweather 8784 1,102.40 1,547.59
   all_8784 lgbm_martina_obs_upper_bound 8784   998.45 1,409.79
   all_8784                 seasonal_avg 8784 2,718.46 3,653.91
   all_8784                        terna 8784   603.93   816.30
common_8783       lgbm_martina_noweather 8783 1,102.52 1,547.68
common_8783 lgbm_martina_obs_upper_bound 8783   998.56 1,409.87
common_8783               seasonal_naive 8783 1,975.14 3,177.19
common_8783                 seasonal_avg 8783 2,718.58 3,654.08
common_8783                        terna 8783   603.98   816.34


## Feature importance

Gain importance of `lgbm_martina_noweather`, the operational version in this notebook. The share is the fraction of total gain on the publication-restricted fit.

The weekly lag `actual_load_mw_lag168h` accounts for 81.0% of the gain, so the model behaves largely as a weekly seasonal naive corrected by calendar features. Terna's forecast remains ahead of `lgbm_martina_noweather` in both tables above, because it uses information that is not in these features. That gap is the main limit of this version.


In [6]:
gain = pd.Series(
    model_noweather.booster_.feature_importance(importance_type="gain"),
    index=COMMON_FEATURES,
    name="gain_share",
).sort_values(ascending=False)
gain = gain / gain.sum()
print(gain.to_string(float_format=lambda value: f"{value:.3f}"))


actual_load_mw_lag168h   0.810
actual_load_mw_lag48h    0.047
is_holiday               0.041
actual_load_mw_lag336h   0.027
dow_sin                  0.020
month                    0.016
is_christmas_period      0.011
hour_cos                 0.010
dow_cos                  0.007
hour_sin                 0.005
is_bridge                0.003
is_summer                0.003
is_holiday_eve           0.001
is_weekend               0.000
